In [2]:
# gan on mnist
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import numpy as np

(x_train,_),(_,_)=mnist.load_data()

#normalize [-1,1]
x_train=(x_train.astype("float32")-127.5)/127.5
x_train=np.expand_dims(x_train, axis=-1)
print(x_train.shape)
#generator

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, Dense, Reshape

def gan():
    model=Sequential()
    model.add(Dense(128, input_dim=100))
    model.add(LeakyReLU())
    model.add(Dense(784, activation='tanh'))
    model.add(Reshape((28,28,1)))
    return model

gen=gan()
gen.summary()
#build discriminator

from tensorflow.keras.layers import Flatten

def discrim():
    model=Sequential()
    model.add(Flatten(input_shape=(28,28,1)))
    model.add(Dense(128))
    model.add(LeakyReLU())
    model.add(Dense(1,activation='sigmoid'))
    return model

disc=discrim()
disc.summary()
#compile

disc.compile(optimizer='adam', loss='binary_crossentropy')
disc.trainable=False
gan_model=Sequential([gen,disc])
disc.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
disc.trainable=True
#train

import matplotlib.pyplot as plt
epochs=50
batch_size=128
half_batch=batch_size//2

d_loss=[]
g_loss=[]

for e in range(epochs):
    idx=np.random.randint(0, x_train.shape[0], half_batch)
    real=x_train[idx]
    noise=np.random.normal(0,1,(half_batch, 100))
    fake=gen.predict(noise, verbose=0)
    d_loss_r=disc.train_on_batch(real, np.ones((half_batch,1)))
    d_loss_f=disc.train_on_batch(fake, np.zeros((half_batch,1)))   
    d = 0.5 * (d_loss_r[0] + d_loss_f[0])   # only loss

    noise=np.random.normal(0,1,(batch_size,100))
    g=float(gan_model.train_on_batch(noise, np.ones((batch_size,1))))

    d_loss.append(d)
    g_loss.append(g)
    print(f"epoch {e+1} | d loss: {d} | g loss: {g}")
plt.plot(d_loss)
plt.plot(g_loss)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend(["disc loss", "gen loss"])
plt.show()
#generate

noise = np.random.normal(0,1,(5,100))
generated_imgs = gen.predict(noise)

# rescale back to [0,1]
generated_imgs = 0.5 * generated_imgs + 0.5

for i in range(5):
    plt.imshow(generated_imgs[i].reshape(28,28), cmap='gray')
    plt.axis('off')
    plt.show()

# evaluation of discriminator

# generate fake images
noise = np.random.normal(0,1,(100,100))
fake_imgs = gen.predict(noise)

# labels
real_labels = np.ones((100,1))
fake_labels = np.zeros((100,1))

# evaluate
d_loss_real = disc.evaluate(x_train[:100], real_labels, verbose=0)
d_loss_fake = disc.evaluate(fake_imgs, fake_labels, verbose=0)

print("Real Accuracy:", d_loss_real)
print("Fake Accuracy:", d_loss_fake)

(60000, 28, 28, 1)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                      │ (None, 128)                 │          12,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_2 (LeakyReLU)            │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 784)                 │         101,136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape_1 (Reshape)                  │ (None, 28, 28, 1)           │               0 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 114,064 (445.56 KB)

 Trainable params: 114,064 (445.56 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)                  │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 128)                 │         100,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_3 (LeakyReLU)            │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 100,609 (393.00 KB)

 Trainable params: 100,609 (393.00 KB)

 Non-trainable params: 0 (0.00 B)

ValueError: You must call `compile()` before using the model.